### **BASIC IMPORTS**

In [1]:
import os
import torch
import yaml
import argparse
from pathlib import Path

### **CONFIGURATION BUILDER**

#### **BUILDING yaml FILES**

In [94]:
# Define path
path = './cf_files/base.yaml'
# Read the yaml file
with open(path, mode = 'r') as file:
    data = yaml.safe_load(file)

# Show data aspect
data

{'MODEL_ATTR': {'vocab_size': 50257,
  'context_length': 1024,
  'n_layers': 12,
  'emb_dim': 768,
  'dropout': 0.1,
  'ffn_hidden_dim': None,
  'moe_mask': {'schedule': 'alternate', 'offset': 2},
  'attention_related': {'num_heads': 12, 'dropout': 0.1, 'qkv_bias': True},
  'moe_related': {'experts': 6, 'k_routing': 1, 'hidden_dim': None}},
 'TRAIN': {'epochs': 100, 'bs': 10, 'lr': 3e-05},
 'EXPCONFIG': {'name': 'E0',
  'init_mode': 'random',
  'load_modality': True,
  'stages': [{'name': 'test_1',
    'groups': [],
    'expert': None,
    'tasks': None,
    'loss': 'ce'}]}}

#### **BUILDING TRAINING CONFIGS BASED ON PARAMS**

In [89]:
class ExperimentConfig():

    # Define constant values
    TASKS = ["add_sub", "mul_div", "alphabetic", "spell", "parity", "first_last"]
    BASE_DIR = 'cf_files'

    def __init__(self, name:str):

        # Define the base data storage
        self.data:dict = None
        # Load yaml file
        self.load_yaml(name)

        # Define stages as None
        self.stages:list[dict] = None
        self.iter_idx:int = 0

        # Build the stages
        self.build_stages()
        self.stop_id = len(self.stages['stages'])

        print('Aspecto de las stages', self.stages)


    def get_train(self)->dict:
        """  
            Function aimed to return train 
            configuration

            Returns:
                train_config(dict):
                    Dictionary containing train 
                    configuration
        """
        return self.data["TRAIN"]

    def __iter__(self):
        return self

    def __next__(self):
        # Check index
        if self.iter_idx >(self.stop_id - 1):
            # Reboot iter idx
            self.iter_idx = 0
            raise StopIteration
        # Return next item 
        value = self.stages['stages'][self.iter_idx]
        # Increase idx
        self.iter_idx+=1
        return value



    def build_stages(self)->None:
        """  
            Function aimed to build stages for 
            our stages_based training
            
            Remember that stages have the following
            format:

            STAGES:
                name: 'E0'
                init_mode: random
                stages: 
                     - name: "test_1"
                      groups:
                     - router
                     - ffn
                      expert: null
                      tasks: null
                      loss: ce

        """ 
        # Get stages data
        stages = self.data['STAGES']    
        # Add expert stages
        expert_stages = [{"name": f"expert_{i}", "groups": ["experts"], "expert": i, "tasks": [task], "loss": "ce"}
                    for i, task in enumerate(self.TASKS)]
        # Modify base stages
        stages["stages"] = expert_stages + stages["stages"]

        # Set values
        self.stages = stages

    @staticmethod
    def file_checker(name:str)->str:
        """
            Function aimed to check file existance
            In case the selected file does not exist,
            a default file will be returned

            Params:
                name(str):
                    The yaml name to validae

            Returns:
                final_path(str):
                    The path for the final file
        """
        # Define mock path
        mock_path = os.path.join(ExperimentConfig.BASE_DIR, name)
        # Check if it exists
        if os.path.exists(mock_path):
            return mock_path

        print(f'The seeked file ({name}) does not exist')
        return  os.path.join(ExperimentConfig.BASE_DIR, 'base.yaml')

    def load_yaml(self, name:str)->dict:
        """  
            Function aimed to conduct yaml file
            loading

            Params:
                name(str):
                    The name of the file to load

            Return:
                data(dict):
                    The dictionary containing retrieved params

            
        """

        # Get path
        path = self.file_checker(name)

        # Load data
        with open(path, mode = 'r') as file:
            self.data = yaml.safe_load(file)


In [90]:
# Define the object
name = 'base.yaml'
configurator = ExperimentConfig(name = name)

# Aspect of the whole yaml data
print('Aspect of the whole yaml:', configurator.data)

# Get train params
train_params = configurator.get_train()
print('Aspect of train hyperparams:', train_params)

# Check iterator
print('Testing Iterator')
for element in configurator:
    print(element)

# Stages building 
configurator.build_stages

Aspecto de las stages {'name': 'E0', 'init_mode': 'random', 'stages': [{'name': 'expert_0', 'groups': ['experts'], 'expert': 0, 'tasks': ['add_sub'], 'loss': 'ce'}, {'name': 'expert_1', 'groups': ['experts'], 'expert': 1, 'tasks': ['mul_div'], 'loss': 'ce'}, {'name': 'expert_2', 'groups': ['experts'], 'expert': 2, 'tasks': ['alphabetic'], 'loss': 'ce'}, {'name': 'expert_3', 'groups': ['experts'], 'expert': 3, 'tasks': ['spell'], 'loss': 'ce'}, {'name': 'expert_4', 'groups': ['experts'], 'expert': 4, 'tasks': ['parity'], 'loss': 'ce'}, {'name': 'expert_5', 'groups': ['experts'], 'expert': 5, 'tasks': ['first_last'], 'loss': 'ce'}, {'name': 'test_1', 'groups': ['router', 'ffn'], 'expert': None, 'tasks': None, 'loss': 'ce'}]}
Aspect of the whole yaml: {'MODEL_ATTR': {'vocab_size': 50257, 'context_length': 1024, 'n_layers': 12, 'emb_dim': 768, 'dropout': 0.1, 'ffn_hidden_dim': None, 'moe_mask': {'schedule': 'alternate', 'offset': 2}, 'attention_related': {'num_heads': 12, 'dropout': 0.1, '

<bound method ExperimentConfig.build_stages of <__main__.ExperimentConfig object at 0x7ea0a93fb560>>